> Steps Required:
1. Get the original sequence 
2. Translate original sequence
3. Trim spacers from original sequence
4. utilize the AnarcII on the sequence
5. Import the CDR3 from the anarchii alligned sequence to the original sequence
6. Profit

In [1]:
from scripts.helpers import codon_dict

def fast_translate(nt_seq:str) -> str:
    length = len(nt_seq)
    seq_2t = nt_seq[:length - (length % 3)]
    aa_seq = []

    for i in range(1, int(len(seq_2t)/3 + 1)):
        try:
            aa_seq.append(codon_dict[seq_2t[i*3-3:i*3]])
        except:
            aa_seq.append("-")

    return "".join(aa_seq)

In [2]:
from scripts.helpers import translateNT
import pandas as pd 
import os 

n = 100

input_path = os.path.join("input", "cleaned_seqs_all_seq_1k.csv")
input_df = pd.read_csv(input_path, index_col=0)
input_totest = input_df.head(n).germline.str.replace("-","")

In [ ]:
from scripts.helpers import translateNT
from anarcii import Anarcii
from scripts.anarcii import aligned_frame
import pandas as pd 
import os  

class AssignCDR3():
    def __init__(self, 
               input_df:pd.DataFrame | str, 
               germline_column:str,
               seq_column:str,
               append_cdr3: tuple = (True, "cdr3_aa"),
               model_seqtype:str = "antibody", 
               model_mode: str = "accuracy"):
        """
        Custom class that takes a NT sequence column from dataframe and assign an AnarcII allighen CDR3 to it.

        input_df: pd.DataFrame | str -> Dataset in pd.DataFrame format or exact string path to the input dataset in CSV format.
        germline_column: str -> string name of the column which contains the germline NT sequence.
        seq_column: str -> string name of the column which contains the sequencing NT sequence.
        append_cdr3: tuple with boolean value and string, if True will try to append the cdr3 aa sequence originated from the specified column into the translated sequence.
        model_seqtype:str -> Which model the anarcii algorithm wil load.
        model_mode:str -> Processing mode of the anarcii algorithm.
        """

        # Defining column names
        self.germline_column = germline_column
        self.seq_column = seq_column
        self.append_cdr3 = append_cdr3

        # Getting CDR3 information if required
        if append_cdr3[0]:
            self.cdr3_aa_col = append_cdr3[1] 


        # Importing the raw data into python
        if isinstance(input_df, pd.DataFrame):
            self.input_df = input_df

        elif isinstance(input_df, str):
            try:
                self.input_df = pd.read_csv(input_df, index_col=0)

            except:
                raise Exception(f"> Invalid input path (under `input_df` argument): `{input_df}`")

        self.anarchii_model = Anarcii(seq_type=model_seqtype, mode=model_mode)


    def anarchii_allign(self):
        """
        Initializing the class, getting the raw data, and alligining the sequence according to  
        the AnacrII algorithm.
        """
        # Getting the germline NT column -> translating to AA
        germline_nt = self.input_df[self.germline_column]
        germline_aa = germline_nt.apply(translateNT,aa_end=104).str.replace("-","").str.replace("X","")

        if self.append_cdr3[0]:
            germline_aa = germline_aa + self.input_df[self.append_cdr3[1]]

        # Creating anarcii results dataframe
        sequence_anarcii = self.anarchii_model.number(germline_aa)
        self.results_df = pd.DataFrame(sequence_anarcii).T

        # inserting alligned sequence to the results dataframe
        results_anarcii = aligned_frame(sequence_anarcii)
        self.results_df.insert(loc=1, 
                               column="sequence", 
                               value= results_anarcii.astype(str).agg("".join, axis=1))
        
        return self.results_df
        
    def implant_cdr3(self):
        """
        Inserting the AnarcII-alligned CDR3 sequence into the ImmuneDB seq (from aa position 104 onward).
        """
        og_upto104 = 

SyntaxError: unterminated string literal (detected at line 51) (379728561.py, line 51)

In [ ]:
anarcii_class = AssignCDR3(input_df.head(10), germline_column="germline", seq_column="sequence")
anarci_alligment = anarcii_class.anarchii_allign()
anarci_implant = 1
